This notebook compares stats for iterativeWGCNA in 8 conditions versus 14 conditions for each strain separately.

In [111]:
import pandas as pd
import numpy as np

In [112]:
file_path = '/Users/annasve/Desktop/article_data/output/iterative_WGCNA/merged_0.2_5_signed/all_modules/NBC_00906_modules_unfiltered.xlsx'
df_8 = pd.read_excel(file_path)

In [113]:
file_path = '/Users/annasve/Desktop/article_data/output/iterative_WGCNA/merged_0.2_5_signed_14/NBC_00906_modules_unfiltered.xlsx'
df_14 = pd.read_excel(file_path)

In [114]:
# 1) Minimal assignment tables
assign_8 = df_8[['Gene', 'Module']].copy()
assign_14 = df_14[['Gene', 'Module']].copy()

# 2) Normalise "unclassified" labels to NaN
unclassified_labels = ['UNCLASSIFIED', 'grey', 'grey60', 'unassigned', '0']

assign_8['Module'] = assign_8['Module'].replace(unclassified_labels, np.nan)
assign_14['Module'] = assign_14['Module'].replace(unclassified_labels, np.nan)

# 3) Rename to avoid confusion after merge
assign_8 = assign_8.rename(columns={'Module': 'Module_8'})
assign_14 = assign_14.rename(columns={'Module': 'Module_14'})


In [115]:
merged = pd.merge(assign_8, assign_14, on='Gene', how='outer')
merged.head()

,Gene,Module_8,Module_14
0,OG694_00005,NaN,NaN
1,OG694_00010,NaN,NaN
2,OG694_00015,P3_I6_M62,NaN
3,OG694_00020,P3_I6_M53,NaN
4,OG694_00025,P3_I6_M53,NaN


In [116]:
# Number of modules (excluding NaN)
n_modules_8  = merged['Module_8'].dropna().nunique()
n_modules_14 = merged['Module_14'].dropna().nunique()

print("Modules in 8-condition run:", n_modules_8)
print("Modules in 14-condition run:", n_modules_14)

# Unclassified genes
n_unclass_8  = merged['Module_8'].isna().sum()
n_unclass_14 = merged['Module_14'].isna().sum()

print("Unclassified genes in 8-condition run:", n_unclass_8)
print("Unclassified genes in 14-condition run:", n_unclass_14)


Modules in 8-condition run: 197
Modules in 14-condition run: 248
Unclassified genes in 8-condition run: 1246
Unclassified genes in 14-condition run: 2046


In [117]:
def classify_row(row):
    m8, m14 = row['Module_8'], row['Module_14']
    if pd.isna(m8) and pd.isna(m14):
        return "unclassified_both"
    if pd.isna(m8) and not pd.isna(m14):
        return "newly_assigned_in_14"
    if not pd.isna(m8) and pd.isna(m14):
        return "lost_module_in_14"
    if m8 == m14:
        return "same_module"
    return "changed_module"

merged['status'] = merged.apply(classify_row, axis=1)
print(merged['status'].value_counts())


status
changed_module          6599
lost_module_in_14       1224
unclassified_both        822
newly_assigned_in_14     424
Name: count, dtype: int64


In [118]:
# Only genes that have some module in at least one run
mask_any = merged[['Module_8', 'Module_14']].notna().any(axis=1)
overlap = pd.crosstab(merged.loc[mask_any, 'Module_8'],
                      merged.loc[mask_any, 'Module_14'])

overlap.head()


Module_14,P10_I3_M1,P10_I3_M2,P1_I14_M1,P1_I14_M10,P1_I14_M11,P1_I14_M12,P1_I14_M13,P1_I14_M14,P1_I14_M15,P1_I14_M16,...,P7_I3_M6,P7_I3_M7,P7_I3_M8,P7_I3_M9,P8_I3_M10,P8_I3_M2,P8_I3_M4,P8_I3_M5,P8_I3_M7,P9_I3_M1
Module_8,,,,,,,,,,,,,,,,,,,,,
P10_I3_M1,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
P10_I3_M2,0,0,6,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
P10_I3_M3,0,0,0,0,0,0,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
P1_I13_M1,0,0,310,0,0,6,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
P1_I13_M10,0,0,0,52,0,0,0,0,6,49,...,0,0,0,0,1,0,0,0,0,0


In [119]:
row_prop = overlap.div(overlap.sum(axis=1), axis=0)  # fractions per 8-module

In [120]:
row_prop

Module_14,P10_I3_M1,P10_I3_M2,P1_I14_M1,P1_I14_M10,P1_I14_M11,P1_I14_M12,P1_I14_M13,P1_I14_M14,P1_I14_M15,P1_I14_M16,...,P7_I3_M6,P7_I3_M7,P7_I3_M8,P7_I3_M9,P8_I3_M10,P8_I3_M2,P8_I3_M4,P8_I3_M5,P8_I3_M7,P9_I3_M1
Module_8,,,,,,,,,,,,,,,,,,,,,
P10_I3_M1,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,1.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
P10_I3_M2,0.0,0.0,0.101695,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
P10_I3_M3,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.073171,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
P1_I13_M1,0.0,0.0,0.500808,0.000000,0.0,0.009693,0.0,0.001616,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
P1_I13_M10,0.0,0.0,0.000000,0.239631,0.0,0.000000,0.0,0.000000,0.027650,0.225806,...,0.0,0.0,0.0,0.0,0.004608,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P7_I3_M14,0.0,0.0,0.500000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
P7_I3_M15,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
P7_I3_M6,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0


In [121]:
row_prop.to_excel('/Users/annasve/Desktop/article_data/output/8_vs_14_conditions/NBC_00906/row_prop.xlsx')

In [122]:
overlap_long = overlap.stack().reset_index()
overlap_long.columns = ['Module_8', 'Module_14', 'n']

# remove zeros
overlap_long = overlap_long[overlap_long['n'] > 0]

# For each 8-module, find best matching 14-module
best_match_8 = (
    overlap_long
    .assign(frac_in_8=lambda d: d['n'] / d.groupby('Module_8')['n'].transform('sum'))
    .sort_values(['Module_8', 'frac_in_8'], ascending=[True, False])
    .groupby('Module_8')
    .head(1)
)

best_match_8.head()


,Module_8,Module_14,n,frac_in_8
237,P10_I3_M1,P7_I3_M6,1,1.000000
279,P10_I3_M2,P1_I14_M5,9,0.152542
516,P10_I3_M3,P1_I14_M37,20,0.487805
743,P1_I13_M1,P1_I14_M1,310,0.500808
991,P1_I13_M10,P1_I14_M10,52,0.239631


In [123]:
best_match_8.to_excel('/Users/annasve/Desktop/article_data/output/8_vs_14_conditions/NBC_00906/best_match_8.xlsx', index = False)

In [124]:
def module_conservation(frac):
    if frac >= 0.7:
        return "conserved"
    elif frac >= 0.3:
        return "partial/split"
    else:
        return "reorganised"

best_match_8['conservation'] = best_match_8['frac_in_8'].apply(module_conservation)
best_match_8['conservation'].value_counts()


conservation
partial/split    112
conserved         41
reorganised       40
Name: count, dtype: int64